
# Predicting the Next Phase of COVID-19 in Kenya — Confirmed Cases (Version 2)

**Methodological revision of the original May 2020 MSc project**

This notebook corrects the confirmed-cases forecasting workflow while preserving the historical information boundary of the original study.

### Historical integrity rules
1. No observations after **14 May 2020** are used to fit the historical model.
2. Model validation uses a genuine temporal holdout:
   - **Training:** through 30 April 2020
   - **Test:** 1–14 May 2020
3. The final model is refit through 14 May 2020 only after validation.
4. The 100-day output is described as a **statistical projection**, not a deterministic epidemic prediction.
5. A dated/local May-2020 JHU snapshot is preferred. A modern archived JHU file may contain retrospective corrections and should not silently replace the historical source.

> **Important:** This Version 2 corrects methodology and reproducibility issues; it does not use knowledge of what happened after 14 May 2020 to tune the original forecast.


In [ ]:

# 1. Imports and configuration
from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

COUNTRY = "Kenya"
HISTORICAL_CUTOFF = pd.Timestamp("2020-05-14")
TRAIN_END = pd.Timestamp("2020-04-30")
TEST_START = pd.Timestamp("2020-05-01")
TEST_END = HISTORICAL_CUTOFF

# Preferred: place the exact CSV snapshot used in 2020 here.
LOCAL_DATA = Path("../data/original/time_series_covid19_confirmed_global.csv")

# Fallback only. This archived file contains the full historical series and
# may include retrospective corrections made after May 2020.
JHU_ARCHIVED_URL = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
    "csse_covid_19_data/csse_covid_19_time_series/"
    "time_series_covid19_confirmed_global.csv"
)

print("Historical cutoff:", HISTORICAL_CUTOFF.date())
print("Training ends:", TRAIN_END.date())
print("Test period:", TEST_START.date(), "to", TEST_END.date())



## 2. Load the JHU confirmed-cases data

For strict reproducibility, use the **same May 2020 CSV snapshot** used in the original research.  
If that file is unavailable, the notebook can fall back to the archived JHU dataset, but the fallback is explicitly flagged because historical values may have been revised retrospectively.


In [ ]:

# 2. Load source data
if LOCAL_DATA.exists():
    source = str(LOCAL_DATA)
    provenance = "Local historical snapshot"
else:
    source = JHU_ARCHIVED_URL
    provenance = "Archived JHU master fallback (verify for retrospective corrections)"

raw = pd.read_csv(source)

print("Data provenance:", provenance)
print("Raw shape:", raw.shape)
raw.head()


In [ ]:

# 3. Extract Kenya explicitly and construct a daily cumulative series
date_columns = raw.columns[4:]

kenya_rows = raw.loc[raw["Country/Region"] == COUNTRY, date_columns]

if kenya_rows.empty:
    raise ValueError(f"{COUNTRY} was not found in the source dataset.")

confirmed = (
    kenya_rows.sum(axis=0)
    .rename("Confirmed")
    .astype(float)
)

confirmed.index = pd.to_datetime(confirmed.index, format="%m/%d/%y")
confirmed = confirmed.sort_index()

# Enforce the May 14, 2020 historical information boundary.
confirmed = confirmed.loc[:HISTORICAL_CUTOFF]

covid = confirmed.to_frame()
covid["Daily_New_Cases"] = covid["Confirmed"].diff().fillna(0)
covid["7_Day_Average"] = covid["Daily_New_Cases"].rolling(7).mean()

print("Series begins:", covid.index.min().date())
print("Series ends:", covid.index.max().date())
print("Confirmed cases at cutoff:", int(covid["Confirmed"].iloc[-1]))
covid.tail(10)



## 3. Exploratory analysis

The original project modeled cumulative confirmed cases. Version 2 preserves that series for comparability, while also inspecting **daily incidence** and a **7-day moving average**, which are more informative about changes in epidemic momentum.


In [ ]:

# 4. Plot cumulative confirmed cases
ax = covid["Confirmed"].plot(figsize=(11, 5), title="Kenya — Cumulative Confirmed COVID-19 Cases")
ax.set_xlabel("Date")
ax.set_ylabel("Confirmed cases")
plt.show()

# Plot daily incidence and 7-day moving average
ax = covid[["Daily_New_Cases", "7_Day_Average"]].plot(
    figsize=(11, 5),
    title="Kenya — Daily Confirmed Cases and 7-Day Moving Average"
)
ax.set_xlabel("Date")
ax.set_ylabel("Daily cases")
plt.show()



## 4. Temporal train/test split

This is the key correction to the original notebook. The model is trained only on observations available through **30 April 2020**.  
The period **1–14 May 2020** is kept completely unseen for evaluation.


In [ ]:

# 5. Create a genuine temporal holdout
train = covid.loc[:TRAIN_END, "Confirmed"].copy()
test = covid.loc[TEST_START:TEST_END, "Confirmed"].copy()

assert train.index.max() < test.index.min()
assert test.index.max() <= HISTORICAL_CUTOFF

print(f"Training observations: {len(train)}")
print(f"Test observations:     {len(test)}")
print(f"Training range: {train.index.min().date()} to {train.index.max().date()}")
print(f"Test range:     {test.index.min().date()} to {test.index.max().date()}")



## 5. Stationarity assessment

ARIMA-family models require explicit consideration of stationarity.  
The Augmented Dickey-Fuller (ADF) test is run on the level series and the first difference to provide evidence for the differencing order.


In [ ]:

# 6. ADF stationarity tests
def adf_summary(series, label):
    x = series.dropna()
    stat, pvalue, usedlag, nobs, critical, _ = adfuller(x, autolag="AIC")
    print(f"{label}")
    print(f"  ADF statistic: {stat:.4f}")
    print(f"  p-value:       {pvalue:.4f}")
    print(f"  observations:  {nobs}")
    print("  critical values:", {k: round(v, 4) for k, v in critical.items()})
    print()

adf_summary(train, "Level series")
adf_summary(train.diff(), "First-differenced series")


In [ ]:

# 7. ACF/PACF diagnostics on the first-differenced training series
diff_train = train.diff().dropna()

fig = plot_acf(diff_train, lags=min(30, len(diff_train) // 2))
plt.show()

fig = plot_pacf(diff_train, lags=min(20, len(diff_train) // 3), method="ywm")
plt.show()



## 6. Baseline forecast

A forecasting model should beat a simple benchmark.  
The baseline used here is **naïve persistence**: every test-day cumulative count is predicted to equal the last cumulative count observed on 30 April 2020.


In [ ]:

# 8. Forecast evaluation utilities
def evaluate(actual, predicted, model_name):
    actual = pd.Series(actual).astype(float)
    predicted = pd.Series(np.asarray(predicted), index=actual.index).astype(float)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    nonzero = actual != 0
    mape = (
        np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero]).mean() * 100
        if nonzero.any() else np.nan
    )

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_%": mape,
    }

naive_forecast = pd.Series(train.iloc[-1], index=test.index)
baseline_metrics = evaluate(test, naive_forecast, "Naïve persistence")
pd.DataFrame([baseline_metrics])



## 7. ARIMA/SARIMA candidate search

The original notebook searched small combinations of \(p,d,q\) and used a seasonal period of 12.  
Version 2 retains a compact search but compares:

- non-seasonal ARIMA;
- SARIMA with **7-day** seasonality (plausible reporting cycle);
- SARIMA with **12-day** seasonality (for comparison with the original study).

Selection is based primarily on **out-of-sample RMSE**, with AIC/BIC and residual diagnostics used as supporting evidence.


In [ ]:

# 9. Candidate model search
p = q = range(0, 2)
d_values = [1]
pdq = list(itertools.product(p, d_values, q))

seasonal_periods = [None, 7, 12]
candidate_results = []
fitted_candidates = {}

for order in pdq:
    for seasonal_period in seasonal_periods:
        if seasonal_period is None:
            seasonal_orders = [(0, 0, 0, 0)]
        else:
            # Compact seasonal grid to reduce overfitting on a short series.
            seasonal_orders = [
                (0, 0, 0, seasonal_period),
                (1, 0, 0, seasonal_period),
                (0, 0, 1, seasonal_period),
                (1, 0, 1, seasonal_period),
            ]

        for seasonal_order in seasonal_orders:
            label = f"SARIMAX{order}x{seasonal_order}"
            try:
                model = sm.tsa.statespace.SARIMAX(
                    train,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    trend=None
                )
                fit = model.fit(disp=False)

                fc = fit.get_forecast(steps=len(test)).predicted_mean
                metrics = evaluate(test, fc, label)

                row = {
                    **metrics,
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "AIC": fit.aic,
                    "BIC": fit.bic,
                }
                candidate_results.append(row)
                fitted_candidates[label] = fit
            except Exception as exc:
                print("Skipped", label, "->", type(exc).__name__)

results_table = (
    pd.DataFrame(candidate_results)
    .sort_values(["RMSE", "MAE", "AIC"])
    .reset_index(drop=True)
)

results_table.head(15)


In [ ]:

# 10. Compare candidate models against the naïve benchmark
comparison = pd.concat(
    [
        pd.DataFrame([baseline_metrics]),
        results_table[["Model", "MAE", "RMSE", "MAPE_%"]].head(10)
    ],
    ignore_index=True
)

comparison


In [ ]:

# 11. Select the best out-of-sample candidate
best_row = results_table.iloc[0]
BEST_ORDER = tuple(best_row["order"])
BEST_SEASONAL_ORDER = tuple(best_row["seasonal_order"])

print("Selected model:", best_row["Model"])
print("Order:", BEST_ORDER)
print("Seasonal order:", BEST_SEASONAL_ORDER)
print(f"Holdout RMSE: {best_row['RMSE']:.2f}")
print(f"Holdout MAE:  {best_row['MAE']:.2f}")
print(f"Holdout MAPE: {best_row['MAPE_%']:.2f}%")
print(f"AIC: {best_row['AIC']:.2f}")
print(f"BIC: {best_row['BIC']:.2f}")


In [ ]:

# 12. Refit selected specification on TRAIN only for holdout visualization
validation_model = sm.tsa.statespace.SARIMAX(
    train,
    order=BEST_ORDER,
    seasonal_order=BEST_SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
)
validation_fit = validation_model.fit(disp=False)
validation_forecast_obj = validation_fit.get_forecast(steps=len(test))
validation_forecast = validation_forecast_obj.predicted_mean
validation_ci = validation_forecast_obj.conf_int()

ax = train.plot(figsize=(11, 5), label="Training")
test.plot(ax=ax, label="Actual holdout")
validation_forecast.plot(ax=ax, label="Forecast")

ax.fill_between(
    test.index,
    validation_ci.iloc[:, 0].to_numpy(),
    validation_ci.iloc[:, 1].to_numpy(),
    alpha=0.2
)

ax.set_title("Temporal Holdout Validation — 1 to 14 May 2020")
ax.set_ylabel("Cumulative confirmed cases")
ax.legend()
plt.show()



## 8. Residual diagnostics

A well-specified time-series model should leave little systematic autocorrelation in the residuals.  
The Ljung-Box test supplements the standard Statsmodels diagnostic plots.


In [ ]:

# 13. Diagnostic plots
validation_fit.plot_diagnostics(figsize=(12, 8))
plt.show()

# Ljung-Box residual autocorrelation test
max_lag = min(14, max(1, len(validation_fit.resid.dropna()) // 4))
lags = sorted(set([min(7, max_lag), max_lag]))

ljung_box = acorr_ljungbox(
    validation_fit.resid.dropna(),
    lags=lags,
    return_df=True
)
ljung_box



## 9. Refit through 14 May 2020 and produce historical forecasts

Only after the model specification has been selected using the holdout do we refit it on **all observations available through 14 May 2020**.

The outputs are separated into:

- **7-day operational forecast**
- **30-day planning forecast**
- **100-day statistical projection**

The 100-day result should not be interpreted as a deterministic epidemiological prediction. It assumes that the statistical structure observed through 14 May 2020 remains sufficiently informative into the future.


In [ ]:

# 14. Final historical fit through May 14, 2020
full_historical = covid.loc[:HISTORICAL_CUTOFF, "Confirmed"]

final_model = sm.tsa.statespace.SARIMAX(
    full_historical,
    order=BEST_ORDER,
    seasonal_order=BEST_SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_fit = final_model.fit(disp=False)

print(final_fit.summary())


In [ ]:

# 15. Forecast helper
def make_forecast(fitted_model, horizon):
    obj = fitted_model.get_forecast(steps=horizon)
    mean = obj.predicted_mean.rename("Forecast")
    ci = obj.conf_int()
    out = pd.DataFrame({
        "Forecast": mean,
        "Lower_95": ci.iloc[:, 0],
        "Upper_95": ci.iloc[:, 1],
    })
    return out

forecast_7 = make_forecast(final_fit, 7)
forecast_30 = make_forecast(final_fit, 30)
projection_100 = make_forecast(final_fit, 100)

print("7-day forecast")
display(forecast_7)

print("30-day forecast")
display(forecast_30)

print("100-day statistical projection")
display(projection_100)


In [ ]:

# 16. Plot the 100-day statistical projection
ax = full_historical.plot(
    figsize=(12, 6),
    label="Observed through 14 May 2020"
)

projection_100["Forecast"].plot(ax=ax, label="100-day statistical projection")
ax.fill_between(
    projection_100.index,
    projection_100["Lower_95"].to_numpy(),
    projection_100["Upper_95"].to_numpy(),
    alpha=0.2
)

ax.axvline(HISTORICAL_CUTOFF, linestyle="--")
ax.set_title("Kenya COVID-19 Confirmed Cases — 100-Day Statistical Projection")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative confirmed cases")
ax.legend()
plt.show()



## 10. Interpretation and limitations

### What this model estimates
The model forecasts the statistical trajectory of **reported cumulative confirmed cases** in Kenya based on observations available through 14 May 2020.

### What it does not estimate
It does **not** directly estimate:
- total underlying infections;
- the causal effect of curfews, travel restrictions, school closures, or other interventions;
- changes in testing eligibility or testing capacity;
- behavioural changes;
- disease-transmission parameters such as the effective reproduction number.

### Principal limitations
1. **Reported cases are not equivalent to true infections.**
2. **Testing and reporting practices can alter the observed curve.**
3. **The historical series is short**, especially relative to a 100-day projection horizon.
4. **ARIMA/SARIMA is statistical rather than mechanistic epidemiological modelling.**
5. **Long-horizon uncertainty is substantial** and confidence intervals should be interpreted accordingly.
6. If the archived JHU fallback dataset is used instead of the original May-2020 snapshot, the input may include **retrospective historical corrections**.

### Next phase
Freeze this corrected May-2020 model, retrieve actual observations for the subsequent 100 days, and evaluate forecast degradation at 7-, 14-, 30-, 60-, and 100-day horizons without retuning the historical model.
